In [1]:
import pandas as pd
from IPython.display import display, Math

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import numpy as np
import scipy.sparse as sp

from nltk.text import TextCollection

Инициализируем массив ```texts```, который содержит в себе 3 элемента - 3 'синтетических' текста

In [2]:
texts = [u'мама мама мама мыла рама',
         u'рама это рама все просто',
         u'очень просто']

# Считаем частотность

Без использования дополнительных библиотек посчитаем частотность каждого слова по всем текстам

In [3]:
word_to_number = {}
for text in texts:
    for word in text.split():
        word_to_number[word] = word_to_number.get(word, 0) + 1

Для красивой визуализации использум pandas, не обязательно, но наглядно

In [4]:
pd.DataFrame(list(word_to_number.items()))

,0,1
0,мама,3
1,мыла,1
2,рама,3
3,это,1
4,все,1
5,просто,2
6,очень,1


Без использования дополнительных библиотек посчитаем частотность каждого слова по каждому тексту

In [5]:
word_to_number_for_each_texts = {}
for text in texts:
    word_to_number_for_each_texts[text] = {}
    for word in text.split():
        if word_to_number_for_each_texts[text].get(word):
            word_to_number_for_each_texts[text][word] += 1
        else:
            word_to_number_for_each_texts[text][word] = 1

Для красивой визуализации использум pandas, не обязательно, но наглядно (в этот раз для красивой визуализации нужно заморочиться, но мы не будем заморачиваться)

In [6]:
pd.DataFrame(list(word_to_number_for_each_texts.items()))

,0,1
0,мама мама мама мыла рама,"{'мама': 3, 'мыла': 1, 'рама': 1}"
1,рама это рама все просто,"{'рама': 2, 'это': 1, 'все': 1, 'просто': 1}"
2,очень просто,"{'очень': 1, 'просто': 1}"


In [7]:
pd.DataFrame(list(word_to_number_for_each_texts.values())).fillna(0)

,мама,мыла,рама,это,все,просто,очень
0,3.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,2.0,1.0,1.0,1.0,0.0
2,0.0,0.0,0.0,0.0,0.0,1.0,1.0


А теперь воспользуемся библиотекой ```sklearn```

In [8]:
count_vect = CountVectorizer()
temp_matrix = count_vect.fit_transform(texts) # temp_matrix эта промежуточная матрица, понадобится в следующем кейсе,
                                              # для вычисления tf-idf
matrix_counts = temp_matrix.toarray()         # в данной матрице хранятся 

In [9]:
[ x for x in 
sorted (list(count_vect.vocabulary_.items()), key = lambda x: x[1])
]

[('все', 0),
 ('мама', 1),
 ('мыла', 2),
 ('очень', 3),
 ('просто', 4),
 ('рама', 5),
 ('это', 6)]

In [10]:
temp_matrix

<3x7 sparse matrix of type '<class 'numpy.int64'>'
	with 9 stored elements in Compressed Sparse Row format>

Визуализируем красиво ```matrix_counts```

In [11]:
words = [x[0] for x in sorted(count_vect.vocabulary_.items(), key=lambda x: x[1])] # список слов, 
                                                                                   # чтобы сделать красивую шапку
pd.DataFrame(matrix_counts, columns=words)  # при создании DataFrame передадим подготовленный список слов

,все,мама,мыла,очень,просто,рама,это
0,0,3,1,0,0,1,0
1,1,0,0,0,1,2,1
2,0,0,0,1,1,0,0


In [12]:
#tf-idf слова ПРОСТО
import math
1*math.log(3/2)

0.4054651081081644

# Что такое TF-IDF?

## TF - term frequency

1) Просто частотность, что мы считали выше (https://en.wikipedia.org/wiki/Tf–idf)

In [13]:
display(Math(r'f_{t,d}'))

<IPython.core.display.Math object>

2) TF по версии российской википедии (https://ru.wikipedia.org/wiki/TF-IDF) (!!!)

In [14]:
display(Math(r'\mathrm{tf}(t,d) = \frac{f_{t,d}}{\sum_k f_{t_k,d}}'))

<IPython.core.display.Math object>

3) Бинарная встречаемость. *0 - не было слово в тексте, 1 - было слово в тесте*

In [15]:
display(Math(r'\mathrm{tf}(t,d) = \left\{ \begin{matrix} 0, f_{t,d} = 0 \\ 1, f_{t,d} ≥ 1 \end{matrix} \right.'))

<IPython.core.display.Math object>

4) Логорифм от частоты +1

In [16]:
display(Math(r'\mathrm{tf}(t,d) = \log(1 + f_{t,d})'))

<IPython.core.display.Math object>

5) Нормированная частота (double normalization)

In [17]:
display(Math(r'K + (1 - K) \frac { f_{t,d} }{\max({f_{t_i,d}})}'))

<IPython.core.display.Math object>

Подробности https://en.wikipedia.org/wiki/Tf–idf

## IDF - inverse document frequency

1) 'Стандарт', логарифм отношения числа всех документов к числу документов, содержащих данное слово

In [18]:
display(Math(r'\mathrm{idf}(t,D)=\log\frac{N}{|\{d\in D:t\in d\}|}\
             =\log\frac{|D|}{|(d_{i}\supset t_{i})|}=\log\frac{N}{n_t}'))

<IPython.core.display.Math object>

2) Сглаженный IDF

In [19]:
display(Math(r'\mathrm{idf}(t,D)=\log(1+\frac{N}{n_t})'))

<IPython.core.display.Math object>

3) IDF с использованием максимального значения DF

In [20]:
display(Math(r'\log\left(1 + \frac {\max_t n_t} {n_t}\right)'))

<IPython.core.display.Math object>

4) Вероятностная обратная встречаемость

In [21]:
display(Math(r'\log\frac {N - n_t} {n_t}'))

<IPython.core.display.Math object>

# Различные библиотеки для вычисления TF-IDF

## sklearn

Документация http://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer

Исходный код https://github.com/scikit-learn/scikit-learn/blob/bb592f3865f02f1d6bf9dedce1a2554fa0ada800/sklearn/feature_extraction/text.py#L901

In [22]:
tfidf_transformer = TfidfTransformer()  # обратить внимание на ```norm``` и ```smooth_idf```
matrix_tfidf = tfidf_transformer.fit_transform(temp_matrix).toarray()
pd.DataFrame(matrix_tfidf, columns=words)

,все,мама,мыла,очень,просто,рама,это
0,0.000000,0.922383,0.307461,0.000000,0.000000,0.233832,0.000000
1,0.452123,0.000000,0.000000,0.000000,0.343851,0.687703,0.452123
2,0.000000,0.000000,0.000000,0.795961,0.605349,0.000000,0.000000


idf = log(N/n)
idf = MORMALIZE(log(N/(n)+1)+1)

In [23]:
tfidf_transformer = TfidfTransformer(norm=None, smooth_idf=False)  # обратить внимание на ```norm``` и ```smooth_idf```
matrix_tfidf = tfidf_transformer.fit_transform(temp_matrix).toarray()
pd.DataFrame(matrix_tfidf, columns=words)

,все,мама,мыла,очень,просто,рама,это
0,0.000000,6.295837,2.098612,0.000000,0.000000,1.405465,0.000000
1,2.098612,0.000000,0.000000,0.000000,1.405465,2.810930,2.098612
2,0.000000,0.000000,0.000000,2.098612,1.405465,0.000000,0.000000


In [24]:
pd.DataFrame([tfidf_transformer.idf_], columns=words)  # построим только ```idf```

,все,мама,мыла,очень,просто,рама,это
0,2.098612,2.098612,2.098612,2.098612,1.405465,1.405465,2.098612


## NLTK

Исходный код https://github.com/nltk/nltk/blob/7ba46b9d52ed0c03bf806193f38d8c0e9bd8a9b4/nltk/text.py#L531

In [40]:
import nltk

In [41]:
mytexts = TextCollection(texts)

In [42]:
mytexts.tf(u'мама', texts[0])

0.125

In [44]:
mytexts.tf(u'мама', 'мама мама') #делит частоту слов на длину текста в символах

0.2222222222222222

In [43]:
2/9

0.2222222222222222

In [45]:
3/len('корабли лавировали лавировали да не вылавировали')

0.0625

In [46]:
mytexts.tf(u'лавировали', 'корабли лавировали лавировали да не вылавировали')

0.0625